# 03b: Joren et al. Labels (Colab CPU only)

Judge 1,500 questions with Joren et al. (ICLR 2025, App. C.1) sufficient-context prompt using Groq API.

**Test mode:** N_QUESTIONS = 20 (change to None for full 1,500).

In [ ]:
# TEST RUN: set to 20. For full run, set to None.
N_QUESTIONS = 20

In [ ]:
import os
import sys
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict
import re

import pandas as pd
import numpy as np
from scipy.stats import contingency

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Colab: {IN_COLAB}")

In [ ]:
# ── Setup: works on a laptop (VS Code) and on Colab ───────────────────────
import os, sys, json, subprocess
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    # Colab: get the code from GitHub (push your latest commit first) and mount Drive
    REPO, BRANCH = "/content/HALT-RAG", "lift_signal"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Shaithra2107/HALT-RAG", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # laptop: this notebook sits in <repo>/notebooks, so the repo is one level up
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path, data_dir
cfg = load_config(os.path.join(REPO, "config.yaml"))
print("running on:", "Colab" if ON_COLAB else "laptop", "| data folder:", data_dir(cfg))

In [ ]:
# Groq config and paths from config.yaml
print(f"Groq limits: RPM={cfg['judge']['groq_rpm_limit']}, TPM={cfg['judge']['groq_tpm_limit']}, TPD={cfg['judge']['groq_tpd_limit']}")

# Reference config model and limits so we don't repeat the dict access everywhere
GROQ_MODEL = cfg['judge']['groq_model']
GROQ_TEMP = cfg['judge']['groq_temperature']
GROQ_MAX_TOKENS = cfg['judge']['groq_max_tokens']
GROQ_RPM = cfg['judge']['groq_rpm_limit']
GROQ_TPM = cfg['judge']['groq_tpm_limit']
GROQ_TPD = cfg['judge']['groq_tpd_limit']
GROQ_RETRY_WAIT = cfg['judge']['groq_retry_wait_secs']
GROQ_MAX_RETRIES = cfg['judge']['groq_max_retries']

In [ ]:
# Read the Joren et al. prompt template
prompt_path = os.path.join(REPO, 'prompts', 'joren_sufficient_context.txt')
with open(prompt_path) as f:
    prompt_template = f.read()

print(f"Prompt loaded ({len(prompt_template)} chars)")
print(f"Template contains: <question>={('<question>' in prompt_template)}, <context>={('<context>' in prompt_template)}, <TIMESTAMP>={('<TIMESTAMP>' in prompt_template)}, TIMESTAMP={('TIMESTAMP' in prompt_template and 'TIMESTAMP' in prompt_template)}")

# Data paths from config.yaml
signals_path = data_path(cfg, "signals_1500")
checkpoint_path = data_path(cfg, "joren_1500_checkpoint")
output_path = data_path(cfg, "joren_1500")

print(f"Signals CSV: {signals_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Output: {output_path}")

In [ ]:
# Load halt_rag_1500_signals.csv
df = pd.read_csv(signals_path)

print(f"Loaded {len(df)} rows from {os.path.basename(signals_path)}")
print(f"Columns: {list(df.columns)}")

# Verify required columns
required_cols = ['id', 'question', 'band'] + [f'ret_passage_{i}' for i in range(1, 6)] + [f'ret_title_{i}' for i in range(1, 6)]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"WARNING: Missing columns: {missing}")
    print(f"Available: {list(df.columns)}")
else:
    print(f"All required columns present")

# Check for y_cov (for agreement calculation)
has_y_cov = 'y_cov' in df.columns
print(f"Has y_cov: {has_y_cov}")

# Limit to N_QUESTIONS if specified
if N_QUESTIONS is not None:
    df = df.iloc[:N_QUESTIONS].copy()
    print(f"Limited to {len(df)} questions for test run")
else:
    print(f"Running on all {len(df)} questions")

In [ ]:
# Initialize tracking
checkpoint_every = 25

results = []  # list of dicts: {id, question, band, y_joren, raw_output, parse_ok, model, timestamp}
parse_failures = []  # unparseable outputs for retry
sample_outputs = []  # first 3 successful outputs for printing

print(f"Checkpoint path: {checkpoint_path}")
print(f"Output path: {output_path}")

# Resume from checkpoint if it exists
done_ids = set()
if os.path.exists(checkpoint_path):
    checkpoint_df = pd.read_csv(checkpoint_path)
    done_ids = set(checkpoint_df['id'].values)
    print(f"Resuming: {len(done_ids)} questions already done")
    # Load the first 3 samples for display
    sample_outputs = checkpoint_df[checkpoint_df['parse_ok'] == True].head(3)['raw_output'].tolist()
else:
    print(f"Starting fresh (no checkpoint found)")

# Filter to questions not yet done
df_todo = df[~df['id'].isin(done_ids)].copy()
print(f"Questions to process: {len(df_todo)}")

In [ ]:
# Initialize Groq client
try:
    from groq import Groq
except ImportError:
    print("Installing groq...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "groq", "-q"])
    from groq import Groq

# Get API key from Colab Secrets (never print)
if ON_COLAB:
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
else:
    groq_api_key = os.environ.get('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError("GROQ_API_KEY not found. Set it in environment or Colab Secrets.")

client = Groq(api_key=groq_api_key)
print(f"Groq client initialized")

In [ ]:
from pathlib import Path
for _n in list(globals()):
    if _n.endswith('_path') and isinstance(globals()[_n], str):
        globals()[_n] = Path(globals()[_n])
print([n for n in globals() if n.endswith('_path')])

In [ ]:
# Resume from checkpoint if it exists
done_ids = set()
if checkpoint_path.exists():
    checkpoint_df = pd.read_csv(checkpoint_path)
    done_ids = set(checkpoint_df['id'].values)
    print(f"Resuming: {len(done_ids)} questions already done")
    # Load the first 3 samples for display
    sample_outputs = checkpoint_df[checkpoint_df['parse_ok'] == True].head(3)['raw_output'].tolist()
else:
    print(f"Starting fresh (no checkpoint found)")

# Filter to questions not yet done
df_todo = df[~df['id'].isin(done_ids)].copy()
print(f"Questions to process: {len(df_todo)}")

In [ ]:
# Initialize Groq client
try:
    from groq import Groq
except ImportError:
    print("Installing groq...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "groq", "-q"])
    from groq import Groq

# Get API key from Colab Secrets (never print)
if IN_COLAB:
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
else:
    groq_api_key = os.environ.get('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError("GROQ_API_KEY not found. Set it in environment or Colab Secrets.")

client = Groq(api_key=groq_api_key)
print(f"Groq client initialized")

In [ ]:
def estimate_tokens(text):
    """
    Rough estimate: 1 token ≈ 4 chars (Groq's tokenizer may differ, but this is conservative)
    """
    return len(text) // 4 + 1

def call_groq(client, prompt, model, temperature, max_tokens, rpm_limit, tpm_limit, tpd_limit, retry_wait_secs, max_retries):
    """
    Call Groq API with rate limiting and error handling using OpenAI-style SDK.
    Returns: (response_text, success, error_msg, usage_info)
    - success=True: got a valid response
    - success=False, error contains 'daily': hit daily limit, stop
    - success=False, error contains 'retry': hit per-minute limit, will retry
    """
    prompt_tokens = estimate_tokens(prompt)
    output_tokens = max_tokens
    total_tokens = prompt_tokens + output_tokens
    
    # Simple rate limiting: wait if needed
    # In production, track tokens across all requests, but for now assume we start fresh
    tokens_per_minute_estimate = total_tokens
    min_wait_secs = max(total_tokens / (tpm_limit / 60), 60 / rpm_limit)
    
    retry_count = 0
    while retry_count < max_retries:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content, True, None, response.usage
        except Exception as e:
            error_str = str(e)
            if '429' in error_str:
                if 'daily' in error_str.lower() or 'quota' in error_str.lower():
                    return None, False, "daily limit reached", None
                else:
                    # Per-minute limit
                    wait_time = retry_wait_secs * (2 ** retry_count)  # exponential backoff
                    print(f"  429 (per-minute): waiting {wait_time}s before retry {retry_count + 1}/{max_retries}...")
                    time.sleep(wait_time)
                    retry_count += 1
                    continue
            else:
                return None, False, f"API error: {error_str[:100]}", None
    
    return None, False, f"max retries ({max_retries}) exceeded", None

print("estimate_tokens and call_groq functions defined")

In [ ]:
def estimate_tokens(text):
    """
    Rough estimate: 1 token ≈ 4 chars (Groq's tokenizer may differ, but this is conservative)
    """
    return len(text) // 4 + 1

def call_groq(client, prompt, model, temperature, max_tokens, rpm_limit, tpm_limit, tpd_limit, retry_wait_secs, max_retries):
    """
    Call Groq API with rate limiting and error handling.
    Returns: (response_text, success, error_msg)
    - success=True: got a valid response
    - success=False, error contains 'daily': hit daily limit, stop
    - success=False, error contains 'retry': hit per-minute limit, will retry
    """
    prompt_tokens = estimate_tokens(prompt)
    output_tokens = max_tokens
    total_tokens = prompt_tokens + output_tokens
    
    # Simple rate limiting: wait if needed
    # In production, track tokens across all requests, but for now assume we start fresh
    tokens_per_minute_estimate = total_tokens
    min_wait_secs = max(total_tokens / (tpm_limit / 60), 60 / rpm_limit)
    
    retry_count = 0
    while retry_count < max_retries:
        try:
            response = client.messages.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.content[0].text, True, None
        except Exception as e:
            error_str = str(e)
            if '429' in error_str:
                if 'daily' in error_str.lower() or 'quota' in error_str.lower():
                    return None, False, "daily limit reached"
                else:
                    # Per-minute limit
                    wait_time = retry_wait_secs * (2 ** retry_count)  # exponential backoff
                    print(f"  429 (per-minute): waiting {wait_time}s before retry {retry_count + 1}/{max_retries}...")
                    time.sleep(wait_time)
                    retry_count += 1
                    continue
            else:
                return None, False, f"API error: {error_str[:100]}"
    
    return None, False, f"max retries ({max_retries}) exceeded"

print("estimate_tokens and call_groq functions defined")

In [ ]:
# Main loop
print(f"\nProcessing {len(df_todo)} questions...\n")

tokens_used_today = 0  # rough tracking
daily_limit_hit = False
timestamp_str = '2023-11-01'
api_error_ids = []  # track questions that hit API errors (to retry later)

for idx, (_, row) in enumerate(df_todo.iterrows(), 1):
    q_id = row['id']
    question = row['question']
    band = row['band']
    
    # Get 5 passages and titles
    passages = [row.get(f'ret_passage_{i}', '') for i in range(1, 6)]
    titles = [row.get(f'ret_title_{i}', '') for i in range(1, 6)]
    
    # Format prompt
    prompt, format_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
    if prompt is None:
        print(f"[{idx}/{len(df_todo)}] ID {q_id}: format error: {format_err}")
        results.append({
            'id': q_id,
            'question': question,
            'band': band,
            'y_joren': None,
            'raw_output': '',
            'parse_ok': False,
            'model': GROQ_MODEL,
            'timestamp': datetime.now(timezone.utc).isoformat(),
        })
        continue
    
    # Call Groq
    response, success, error, usage = call_groq(
        client, prompt,
        model=GROQ_MODEL,
        temperature=GROQ_TEMP,
        max_tokens=GROQ_MAX_TOKENS,
        rpm_limit=GROQ_RPM,
        tpm_limit=GROQ_TPM,
        tpd_limit=GROQ_TPD,
        retry_wait_secs=GROQ_RETRY_WAIT,
        max_retries=GROQ_MAX_RETRIES,
    )
    
    if not success:
        if 'daily' in error:
            print(f"\n*** DAILY LIMIT REACHED ***")
            print(f"Questions processed: {len([r for r in results if r['y_joren'] is not None or r['parse_ok']])}")
            print(f"Resume by running this notebook again (checkpoint saved).")
            daily_limit_hit = True
            break
        else:
            print(f"[{idx}/{len(df_todo)}] ID {q_id}: API error: {error}")
            # Track API error for retry, but do NOT save to results yet
            api_error_ids.append((q_id, question, band))
            continue
    
    # Parse response
    label, parse_ok = parse_sufficient_context(response)
    if not parse_ok:
        print(f"[{idx}/{len(df_todo)}] ID {q_id}: unparseable (will retry once)")
        parse_failures.append((q_id, question, band, response))
    
    # Store result (only if we got a real response, parsed or not)
    results.append({
        'id': q_id,
        'question': question,
        'band': band,
        'y_joren': label if parse_ok else None,
        'raw_output': response,
        'parse_ok': parse_ok,
        'model': GROQ_MODEL,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    })
    
    if parse_ok and len(sample_outputs) < 3:
        sample_outputs.append(response)
    
    if idx % 10 == 0:
        print(f"[{idx}/{len(df_todo)}] Processed {idx} questions, {sum(1 for r in results if r['parse_ok'])} parsed")
    
    # Save checkpoint every 25 questions (only rows with real responses, not API errors)
    if idx % checkpoint_every == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv(checkpoint_path, index=False)
        print(f"  Checkpoint saved: {len(checkpoint_df)} rows with responses")

print(f"\nLoop finished. Total with responses: {len(results)}")
if api_error_ids:
    print(f"API errors encountered: {len(api_error_ids)} (will retry)")

In [ ]:
# Retry unparseable responses and API errors
all_to_retry = parse_failures + [(q_id, question, band, None) for q_id, question, band in api_error_ids]
print(f"\nRetrying {len(all_to_retry)} failed questions ({len(parse_failures)} unparseable + {len(api_error_ids)} API errors)...\n")

for q_id, question, band, original_response in all_to_retry:
    if daily_limit_hit:
        break
    
    # Format prompt and call Groq again
    passages = [df[df['id'] == q_id].iloc[0].get(f'ret_passage_{i}', '') for i in range(1, 6)]
    titles = [df[df['id'] == q_id].iloc[0].get(f'ret_title_{i}', '') for i in range(1, 6)]
    
    prompt, format_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
    if prompt is None:
        print(f"ID {q_id}: still format error, skipping")
        continue
    
    response, success, error, usage = call_groq(
        client, prompt,
        model=GROQ_MODEL,
        temperature=GROQ_TEMP,
        max_tokens=GROQ_MAX_TOKENS,
        rpm_limit=GROQ_RPM,
        tpm_limit=GROQ_TPM,
        tpd_limit=GROQ_TPD,
        retry_wait_secs=GROQ_RETRY_WAIT,
        max_retries=GROQ_MAX_RETRIES,
    )
    
    if not success:
        if 'daily' in error:
            print(f"Daily limit hit during retry, stopping")
            daily_limit_hit = True
            break
        else:
            print(f"ID {q_id}: retry failed ({error}), skipping")
            continue
    
    # Try to parse again
    label, parse_ok = parse_sufficient_context(response)
    
    # Check if this question already has a result (from API error retry)
    existing = [i for i, r in enumerate(results) if r['id'] == q_id]
    if existing:
        # Update existing result
        idx = existing[0]
        results[idx]['y_joren'] = label if parse_ok else None
        results[idx]['raw_output'] = response
        results[idx]['parse_ok'] = parse_ok
        results[idx]['timestamp'] = datetime.now(timezone.utc).isoformat()
        status = "parsed successfully" if parse_ok else "still unparseable"
        print(f"ID {q_id}: retry {status} (updated existing result)")
        if parse_ok and len(sample_outputs) < 3:
            sample_outputs.append(response)
    else:
        # New result (unparseable case)
        results.append({
            'id': q_id,
            'question': question,
            'band': band,
            'y_joren': label if parse_ok else None,
            'raw_output': response,
            'parse_ok': parse_ok,
            'model': GROQ_MODEL,
            'timestamp': datetime.now(timezone.utc).isoformat(),
        })
        status = "parsed successfully" if parse_ok else "still unparseable"
        print(f"ID {q_id}: retry {status} (new result)")
        if parse_ok and len(sample_outputs) < 3:
            sample_outputs.append(response)

print(f"\nRetry complete.")

In [ ]:
# Retry unparseable responses (once)
print(f"\nRetrying {len(parse_failures)} unparseable responses...\n")

for q_id, question, band, original_response in parse_failures:
    if daily_limit_hit:
        break
    
    # Format prompt and call Groq again
    passages = [df[df['id'] == q_id].iloc[0].get(f'ret_passage_{i}', '') for i in range(1, 6)]
    titles = [df[df['id'] == q_id].iloc[0].get(f'ret_title_{i}', '') for i in range(1, 6)]
    
    prompt, format_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
    if prompt is None:
        print(f"ID {q_id}: still format error, skipping")
        continue
    
    response, success, error = call_groq(
        client, prompt,
        model=GROQ_MODEL,
        temperature=GROQ_TEMP,
        max_tokens=GROQ_MAX_TOKENS,
        rpm_limit=GROQ_RPM,
        tpm_limit=GROQ_TPM,
        tpd_limit=GROQ_TPD,
        retry_wait_secs=GROQ_RETRY_WAIT,
        max_retries=GROQ_MAX_RETRIES,
    )
    
    if not success:
        if 'daily' in error:
            print(f"Daily limit hit during retry, stopping")
            daily_limit_hit = True
            break
        else:
            print(f"ID {q_id}: retry failed, keeping original")
            continue
    
    # Try to parse again
    label, parse_ok = parse_sufficient_context(response)
    if parse_ok:
        # Find and update the result
        for r in results:
            if r['id'] == q_id:
                r['y_joren'] = label
                r['raw_output'] = response
                r['parse_ok'] = True
                r['timestamp'] = datetime.utcnow().isoformat()
                print(f"ID {q_id}: retry successful")
                if len(sample_outputs) < 3:
                    sample_outputs.append(response)
                break
    else:
        print(f"ID {q_id}: still unparseable after retry")

print(f"\nRetry complete.")

In [ ]:
# Save final output
results_df = pd.DataFrame(results)
results_df.to_csv(output_path, index=False)
print(f"Saved {len(results_df)} results to {os.path.basename(output_path)}")

# Also save checkpoint for next resume
results_df.to_csv(checkpoint_path, index=False)
print(f"Checkpoint updated")

In [ ]:
# Compute agreement with y_cov (if available)
if has_y_cov:
    # Merge results with y_cov
    merged = results_df.merge(df[['id', 'y_cov']], on='id', how='left')
    
    # Filter to rows with both labels and parse_ok=True
    valid = merged[(merged['y_joren'].notna()) & (merged['y_cov'].notna()) & (merged['parse_ok'] == True)].copy()
    
    if len(valid) > 0:
        # 2x2 contingency table
        from sklearn.metrics import confusion_matrix, cohen_kappa_score
        
        cm = confusion_matrix(valid['y_cov'], valid['y_joren'], labels=[0, 1])
        accuracy = (valid['y_cov'] == valid['y_joren']).sum() / len(valid)
        kappa = cohen_kappa_score(valid['y_cov'], valid['y_joren'])
        
        print("\n" + "="*60)
        print("Agreement: y_joren vs y_cov")
        print("="*60)
        print(f"\n2×2 Contingency Table (y_cov rows × y_joren cols):")
        print(f"              y_joren=0  y_joren=1")
        print(f"y_cov=0       {cm[0, 0]:6d}     {cm[0, 1]:6d}")
        print(f"y_cov=1       {cm[1, 0]:6d}     {cm[1, 1]:6d}")
        print(f"\nOverall:")
        print(f"  Agreement: {accuracy:.1%}")
        print(f"  Cohen's κ: {kappa:.3f}")
        print(f"  N: {len(valid)}")
        
        # By band
        for band_val in ['head', 'tail']:
            band_valid = valid[valid['band'] == band_val]
            if len(band_valid) > 0:
                acc = (band_valid['y_cov'] == band_valid['y_joren']).sum() / len(band_valid)
                kappa_band = cohen_kappa_score(band_valid['y_cov'], band_valid['y_joren'])
                print(f"\n{band_val.upper()}:")
                print(f"  Agreement: {acc:.1%}")
                print(f"  Cohen's κ: {kappa_band:.3f}")
                print(f"  N: {len(band_valid)}")
    else:
        print("No valid rows for agreement calculation (need both y_cov and y_joren parsed)")
else:
    print("y_cov not available, skipping agreement calculation")

In [ ]:
# Print stats
parsed_count = results_df['parse_ok'].sum()
parse_rate = parsed_count / len(results_df) if len(results_df) > 0 else 0

print("\n" + "="*60)
print("Labeling Stats")
print("="*60)
print(f"Total questions: {len(results_df)}")
print(f"Successfully parsed: {parsed_count} ({parse_rate:.1%})")
print(f"Unparseable: {len(results_df) - parsed_count}")

if parsed_count > 0:
    parsed_df = results_df[results_df['parse_ok'] == True]
    pos_count = (parsed_df['y_joren'] == 1).sum()
    neg_count = (parsed_df['y_joren'] == 0).sum()
    print(f"\nLabels (parsed only):")
    print(f"  Sufficient Context = 1: {pos_count} ({pos_count/len(parsed_df):.1%})")
    print(f"  Sufficient Context = 0: {neg_count} ({neg_count/len(parsed_df):.1%})")
    
    # By band
    for band_val in ['head', 'tail']:
        band_df = parsed_df[parsed_df['band'] == band_val]
        if len(band_df) > 0:
            band_pos = (band_df['y_joren'] == 1).sum()
            print(f"\n{band_val.upper():5s}: sufficient={band_pos}/{len(band_df)} ({band_pos/len(band_df):.1%})")

In [ ]:
# Print 3 sample raw outputs (for test run verification)
if N_QUESTIONS is not None and len(sample_outputs) > 0:
    print("\n" + "="*60)
    print(f"Sample Raw Outputs ({min(3, len(sample_outputs))} of first {len(results_df)} questions)")
    print("="*60)
    for i, output in enumerate(sample_outputs[:3], 1):
        print(f"\n--- Sample {i} ---")
        print(output[:500] if len(output) > 500 else output)
        if len(output) > 500:
            print(f"... (truncated, total {len(output)} chars)")

In [ ]:
print("\nDone.")
if daily_limit_hit:
    print("Daily quota hit. Checkpoint saved. Run this notebook again to resume.")
else:
    print(f"All questions processed. Output: {output_path.name}")